In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = ""
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

**LIBRARIES**

In [ ]:
pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv langchain-huggingface langchain-text-splitters langchain_google_genai

In [114]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1.a Indexing (Document-ingestion)

In [ ]:
video_id = "Gfr50f6ZBvo"
try:
  api = YouTubeTranscriptApi()
  transcript_list = api.fetch(video_id, languages =["en"])
  youtube_transcript = ' '.join(chunk.text for chunk in transcript_list)
  print(youtube_transcript)
except TranscriptsDisabled:
  print("transcript not found")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

# Step 1b- Indexing (Text_Spliter)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.create_documents([youtube_transcript])

In [ ]:
print(len(chunks))

168


In [ ]:
chunks[167]

Document(metadata={}, page_content='demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time')

# Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)



In [ ]:
embedding = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
vectore_store = FAISS.from_documents(chunks, embedding)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
len(vectore_store.index_to_docstore_id)

168

In [ ]:
vectore_store.index_to_docstore_id

{0: 'e7a317b0-069a-49cb-9df1-73c3414b6db6',
 1: '8eeccfd0-a54a-4fb7-9582-89347aa4c24b',
 2: 'ab99ff51-d341-49c6-8034-5b06246b3d7f',
 3: '697d0a1b-49be-4c80-b098-0412f1092dfb',
 4: '3a3ef8e7-9e69-4ee8-9366-795230b6f663',
 5: 'b746aa2b-effa-4570-a989-92a83550afac',
 6: 'ae11003c-564c-4600-bdf3-e356f9666396',
 7: '0185b31d-8f32-4516-bf6a-6a19f2f26702',
 8: 'd96230bb-0de7-4e72-ae79-87cf162b6ebe',
 9: 'c403c829-4455-40d5-ac44-631b61fdee56',
 10: '63a89354-c429-4ae6-a659-403d4433e1cf',
 11: 'f30d73d1-8439-4ad2-a1b2-dc8d2380bbea',
 12: '94a4d0c0-9819-4c71-ba05-93b1ac291da5',
 13: '401552a1-baa5-4ff0-94fc-05e76cff8367',
 14: '6151941a-1ba2-421e-80d3-8b0a2d8a8155',
 15: '7955c5ab-1137-4354-85ca-3cb8c5c3d7bb',
 16: '219a3db2-79f2-4448-8a39-3ac1d6b98b6c',
 17: 'a50c4e41-56ba-41a8-9062-f8e1b38dff72',
 18: '4468c1a2-19c4-4130-a065-21c75e99ac6a',
 19: 'b3e06475-1782-4604-89e6-e67a0407ef8d',
 20: 'e7fa15ae-70cc-4873-bca0-cc34f05750d5',
 21: '9127c1c4-b048-4c75-8e98-09d49e2c4093',
 22: 'bcf7e0bc-f7b2-

In [ ]:
vectore_store.get_by_ids(['d171ff04-ab9a-4273-8ddf-553e2b2a7c93'])

[Document(id='d171ff04-ab9a-4273-8ddf-553e2b2a7c93', metadata={}, page_content="will be a day when ai system can't be denied that it earned that nobel prize do you think we'll see that in 21st century it depends what type of ais we end up building right whether they're um you know goal seeking agents who specifies the goals uh who comes up with the hypotheses who you know who determines which problems to tackle right so i think it's about an announcement yeah so it's announcing the results exactly as part of it um so i think right now of course it's it's it's it's amazing human ingenuity that's behind these systems and then the system in my opinion is just a tool you know it'd be a bit like saying with galileo and his telescope you know the ingenuity the the the credit should go to the telescope i mean it's clearly galileo building the tool which he then uses so i still see that in the same way today even though these tools learn for themselves um they're i think i think of things like

## Step 2 - Retrieval


In [ ]:
retriever = vectore_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a2d990f6350>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke("what is deepmind")

[Document(id='8e613acb-e329-4939-991e-9538cf5851e0', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

# **STEP-3 Augmentation**

In [ ]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash", temperature = 0.2)

In [ ]:
prompt = PromptTemplate(
    template = """
You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
""",
    input_variables = ["context", "question"])

In [ ]:
question = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"

In [ ]:
reteriever_docs = retriever.invoke(question)

In [ ]:
reteriever_docs

[Document(id='39582b08-5315-4909-a678-4acc7bf21dba', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [ ]:
context_text = "\n\n".join([doc.page_content for doc in reteriever_docs])

In [ ]:
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text="\nYou are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look a

## Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
answer.content

"Yes, the topic of nuclear fusion is discussed in the transcript.\n\nWhat was discussed:\n*   The speaker's team collaborated with EPFL in Switzerland, using their test reactor for experiments.\n*   They identify bottleneck problems in fusion that are amenable to AI methods.\n*   Fusion has many challenges, primarily in physics, material science, and engineering.\n*   They applied deep reinforcement learning (deep RL) to control high-temperature plasmas, specifically holding the plasma in specific shapes for a record amount of time. This work was published in a Nature paper.\n*   They are now looking for the next problem to tackle in the fusion area.\n*   The speaker believes AI can help accelerate fusion, which is seen as a crucial area for energy and climate solutions."

# Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

In [104]:
def format_doc(reteriever_docs):
  context_text = "\n\n".join(doc.page_content for doc in reteriever_docs)
  return context_text

In [110]:
parallel_chain = RunnableParallel(
    {
        "context": retriever | RunnableLambda(format_doc),
        "question": RunnablePassthrough()
    }
)

In [111]:
parallel_chain.invoke("who is deemis")

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [115]:
parser = StrOutputParser()

In [116]:
chain = parallel_chain | prompt | llm | parser

In [118]:
chain.invoke("Can you summarize the video")

"The transcript features a conversation between Lex and Demas. They discuss the need for more fundamental and simpler explanations of physics to address long-standing mysteries like consciousness, life, and gravity, as the standard model of physics is considered insufficient. The conversation also explores the idea that a key sign of intelligence, both human and artificial, is the ability to explain complex topics clearly and simply, referencing Richard Feynman. The speakers touch upon the history of AI in chess, mentioning early programs by Claude Shannon and Alan Turing, and the Deep Blue vs. Kasparov match, with the speaker expressing greater admiration for Kasparov's human mind."